# QLoRA fine-tuning Pythia-1.4B step50000 on 100k OpenHermes-2.5 examples

This Kaggle-ready notebook upgrades the previous experiment in two ways:

- **Bigger model:** `EleutherAI/pythia-1.4b` instead of `pythia-410m`.
- **More instruction data:** `teknium/OpenHermes-2.5`, which Hugging Face lists as a 1M-row instruction/chat dataset.

Because 1.4B full-parameter fine-tuning is much heavier than 410M, this notebook uses **QLoRA**. That keeps the base model loaded in 4-bit precision and trains LoRA adapter weights, making the run much more realistic on Kaggle GPUs.

This version is fixed to **100,000 training examples** plus a 1,000-example validation split. It is the practical Kaggle version after the 1.4B full fine-tune ran out of memory.


## 1. Install dependencies

In [ ]:
%pip install -q -U "transformers==4.57.1" "datasets==4.4.1" "accelerate==1.11.0" "trl==0.24.0" "peft==0.17.1" "bitsandbytes==0.48.2" "huggingface_hub==0.36.0" "sentencepiece"


## 2. Imports and configuration

In [ ]:
import gc
import math
import os
import shutil
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo, login
from kaggle_secrets import UserSecretsClient
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "EleutherAI/pythia-1.4b"
MODEL_REVISION = "step50000"
DATASET_ID = "teknium/OpenHermes-2.5"

OUTPUT_DIR = "/kaggle/working/pythia-1.4b-step50000-openhermes-100k-qlora-runs"
ADAPTER_DIR = "/kaggle/working/pythia-1.4b-step50000-openhermes-100k-qlora-adapter"
MERGED_DIR = "/kaggle/working/pythia-1.4b-step50000-openhermes-100k-qlora-merged"
ZIP_PATH = "/kaggle/working/pythia-1.4b-step50000-openhermes-100k-qlora.zip"
HF_REPO_ID = "shehryars715/pythia-1.4b-step50000-openhermes-100k-qlora"

# Fixed experiment size: 100k training rows plus validation rows.
MAX_TRAIN_EXAMPLES = 100_000
VALIDATION_SIZE = 1_000
SEED = 42
MAX_SEQ_LENGTH = 1024

# Adapter-first is the safest Kaggle default. Set True only if you have enough RAM/VRAM to merge after training.
MERGE_AND_PUSH_FULL_MODEL = False

print(f"Base model: {MODEL_ID} @ {MODEL_REVISION}")
print(f"Dataset: {DATASET_ID}")
print(f"HF output repo: https://huggingface.co/{HF_REPO_ID}")

## 3. Check GPU and sign in to Hugging Face

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not found. In Kaggle, enable GPU under Notebook accelerator.")

print(torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
print(f"Compute capability: {major}.{minor}")
print("QLoRA/bitsandbytes works best on Kaggle T4 or A100. P100/K80 can fail with CUDA kernel-image errors.")

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face using Kaggle secret HF_TOKEN.")
except Exception as exc:
    raise RuntimeError(
        "Could not read Kaggle secret HF_TOKEN. Add it in Kaggle: Add-ons -> Secrets -> HF_TOKEN."
    ) from exc

## 4. Load tokenizer and 4-bit base model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print("Loaded 4-bit model for QLoRA training.")

## 5. Load and format OpenHermes conversations

In [ ]:
def role_name(role):
    role = str(role).lower().strip()
    if role in {"human", "user"}:
        return "user"
    if role in {"gpt", "assistant"}:
        return "assistant"
    if role == "system":
        return "system"
    return role or "user"


def format_openhermes(example):
    conversations = example.get("conversations") or []
    pieces = []

    system_prompt = example.get("system_prompt")
    if system_prompt:
        pieces.append(f"<|system|>\n{str(system_prompt).strip()}\n")

    for turn in conversations:
        if not isinstance(turn, dict):
            continue
        role = role_name(turn.get("from", "user"))
        value = str(turn.get("value", "")).strip()
        if not value:
            continue
        pieces.append(f"<|{role}|>\n{value}\n")

    text = "".join(pieces).strip()
    if text:
        text = text + tokenizer.eos_token
    return {"text": text}

raw_dataset = load_dataset(DATASET_ID, split="train")
print(f"Raw rows: {len(raw_dataset):,}")

formatted = raw_dataset.map(
    format_openhermes,
    remove_columns=raw_dataset.column_names,
    desc="Formatting conversations",
)
formatted = formatted.filter(lambda x: bool(x["text"] and x["text"].strip()), desc="Dropping empty rows")
formatted = formatted.shuffle(seed=SEED)

if MAX_TRAIN_EXAMPLES is None:
    total_needed = len(formatted)
else:
    total_needed = min(len(formatted), MAX_TRAIN_EXAMPLES + VALIDATION_SIZE)

selected = formatted.select(range(total_needed))
validation_size = min(VALIDATION_SIZE, max(1, len(selected) // 100))
validation_dataset = selected.select(range(validation_size))
train_dataset = selected.select(range(validation_size, len(selected)))

print(f"Usable rows: {len(formatted):,}")
print(f"Training rows: {len(train_dataset):,}")
print(f"Validation rows: {len(validation_dataset):,}")
if MAX_TRAIN_EXAMPLES is not None and len(formatted) >= MAX_TRAIN_EXAMPLES + validation_size:
    assert len(train_dataset) == MAX_TRAIN_EXAMPLES, f"Expected exactly {MAX_TRAIN_EXAMPLES:,} training rows, got {len(train_dataset):,}"
print("\nExample:\n", train_dataset[0]["text"][:1500])

## 6. Configure LoRA and training

In [ ]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "query_key_value",
        "dense",
        "dense_h_to_4h",
        "dense_4h_to_h",
    ],
)

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=0.3,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    logging_steps=25,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_num_proc=2,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("Trainable parameters:")
trainer.model.print_trainable_parameters()
effective_batch_size = sft_args.per_device_train_batch_size * sft_args.gradient_accumulation_steps
estimated_optimizer_steps = math.ceil(len(train_dataset) / effective_batch_size)
print(f"Effective batch size: {effective_batch_size}")
print(f"Estimated optimizer steps for one epoch: {estimated_optimizer_steps:,}")

## 7. Quick base-model sample before training

In [ ]:
test_prompts = [
    "Explain photosynthesis in exactly two sentences.",
    "A car travels 120 km in 2 hours. What is its average speed?",
    "Write a Python function that checks if a number is prime.",
]


def make_chat_prompt(question):
    return f"<|user|>\n{question.strip()}\n<|assistant|>\n"


def generate(prompt, max_new_tokens=160):
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        output = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.12,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()

base_rows = []
for q in test_prompts:
    answer = generate(q)  # raw prompt, intentionally not chat formatted
    base_rows.append({"question": q, "base_model_output": answer})
    print("QUESTION:", q)
    print("BASE:", answer[:1000])
    print("-" * 80)

## 8. Train

In [ ]:
train_result = trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

metrics = train_result.metrics
metrics["train_rows"] = len(train_dataset)
metrics["validation_rows"] = len(validation_dataset)
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"Saved LoRA adapter to {ADAPTER_DIR}")

## 9. Evaluate and compare generations

In [ ]:
eval_metrics = trainer.evaluate()
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)
print(eval_metrics)

comparison_rows = []
for row in base_rows:
    q = row["question"]
    finetuned_answer = generate(make_chat_prompt(q))
    comparison_rows.append({
        "question": q,
        "base_model_output": row["base_model_output"],
        "finetuned_model_output": finetuned_answer,
    })
    print("QUESTION:", q)
    print("BASE:", row["base_model_output"][:1000])
    print("FINETUNED:", finetuned_answer[:1000])
    print("-" * 80)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = "/kaggle/working/base_vs_finetuned_outputs_pythia_1_4b_openhermes.csv"
comparison_df.to_csv(comparison_path, index=False)
comparison_df

## 10. Push to Hugging Face

This pushes either a merged full model or the LoRA adapter, depending on `MERGE_AND_PUSH_FULL_MODEL`. If merging runs out of RAM, restart from the saved adapter, set `MERGE_AND_PUSH_FULL_MODEL = False`, and push the adapter only.


In [ ]:
api = HfApi()
create_repo(repo_id=HF_REPO_ID, repo_type="model", private=False, exist_ok=True)

card = f"""---
language: en
base_model: {MODEL_ID}
base_model_revision: {MODEL_REVISION}
datasets:
- {DATASET_ID}
tags:
- text-generation
- causal-lm
- instruction-tuning
- chatbot
- qlora
- pythia
---

# Pythia-1.4B step50000 OpenHermes 100k QLoRA

This model is a QLoRA supervised fine-tune of `{MODEL_ID}` at revision `{MODEL_REVISION}` on 100,000 training examples from `{DATASET_ID}`.

The goal is to scale the earlier 410M Alpaca experiment to a bigger base model and a larger 100k-example instruction/chat subset. The training objective is behavioral: make the early Pythia checkpoint respond in a recognizable assistant/chatbot format instead of raw continuation, echoing, loops, or gibberish-like text.

## Training setup

- Base model: `{MODEL_ID}`
- Revision: `{MODEL_REVISION}`
- Dataset: `{DATASET_ID}`
- Training rows used: `{len(train_dataset):,}`
- Validation rows: `{len(validation_dataset):,}`
- Fine-tuning method: QLoRA / LoRA adapters
- Max sequence length: `{MAX_SEQ_LENGTH}`
- LoRA rank: `{lora_config.r}`
- LoRA alpha: `{lora_config.lora_alpha}`

## Limitations

This is still a small language model by modern chatbot standards. It should not be treated as factually reliable. Success should be measured primarily as instruction-following and chatbot-style behavior, not guaranteed correctness.
"""
Path(ADAPTER_DIR, "README.md").write_text(card, encoding="utf-8")

if MERGE_AND_PUSH_FULL_MODEL:
    print("Merging LoRA adapter into the base model. This can use a lot of CPU RAM.")
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    merged_model = merged_model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(MERGED_DIR)
    Path(MERGED_DIR, "README.md").write_text(card, encoding="utf-8")

    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=MERGED_DIR,
        commit_message="Upload merged Pythia-1.4B OpenHermes 100k QLoRA model",
    )
    shutil.make_archive(ZIP_PATH.replace(".zip", ""), "zip", MERGED_DIR)
else:
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=ADAPTER_DIR,
        commit_message="Upload Pythia-1.4B OpenHermes 100k QLoRA adapter",
    )
    shutil.make_archive(ZIP_PATH.replace(".zip", ""), "zip", ADAPTER_DIR)

api.upload_file(
    repo_id=HF_REPO_ID,
    repo_type="model",
    path_or_fileobj=comparison_path,
    path_in_repo="base_vs_finetuned_outputs.csv",
    commit_message="Add base vs finetuned comparison CSV",
)

print(f"Pushed successfully: https://huggingface.co/{HF_REPO_ID}")
print(f"ZIP backup: {ZIP_PATH}")

## 11. Runtime notes

This notebook is intentionally fixed at `MAX_TRAIN_EXAMPLES = 100_000`.

Use a Kaggle T4 or A100 if possible. If Kaggle assigns a P100/K80 and you see `CUDA error: no kernel image is available for execution on the device`, switch GPU type or restart until you get a T4/A100. QLoRA depends on bitsandbytes 4-bit CUDA kernels.

If you get CUDA out-of-memory errors:

- lower `MAX_SEQ_LENGTH` from `1024` to `768` or `512`,
- keep batch size at `1`,
- keep `MERGE_AND_PUSH_FULL_MODEL = False`,
- push the LoRA adapter first, then merge later on a stronger machine if needed.
